## Patient Journey - Lab Test
#### Data source used
* PHCDW.PHCDW_SANDBOX.IQV_QUEST_LAB
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT 

#### Steps:
1. Query UACR/eGFR lab test claims data after 2017-01-01
2. Get date level lab test stage records (forward fill for between two tests) for each patient
3. Save table to databricks

#### Notes:
1. Time frame: SVC_DT >= '2017-01-01' 
2. GFR test:  upper(LCL_LAB_TEST_NM) like '%GFR%' or upper(LCL_LAB_TEST_NM) like '%GLOMERULAR%' then 'GFR' 
3. UACR test:  upper(LCL_LAB_TEST_NM) like '%ALB%' or upper(LCL_LAB_TEST_NM) like '%RATIO%' then 'UACR'
4. GFR stage: 
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT <15 then 5
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=15 AND MEDIAN_TEST_RESULT <30 then 4
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=30 AND MEDIAN_TEST_RESULT <60 then 3
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=60 AND MEDIAN_TEST_RESULT <90 then 2
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=90 AND  MEDIAN_TEST_RESULT <=150 then 1
5. UACR stage:
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT <30 then 1
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT >=30 and MEDIAN_TEST_RESULT <300 then 2
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT >=300 then 3
6. Check the the end_date of the data, now we set end_date_limit = Max(SVC_DT)

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../00_cohort_config"

In [0]:
# Source: the 3.0 "Pat Journey Lab" query (00_Write_Input_Files), run directly from Snowflake.
# The pre-built kerendia_nba_3_0_lab_query_output table was removed in the 2026-10-26 refresh,
# so this notebook now pulls it itself (same pattern as 03_patient_journey_rx / 04_t2d_flag).
lab_query = """
      SELECT
          PATIENT_ID,
          SVC_DT_STD AS DATE,
          NPI_ID AS NPI_NUMBER,
          TEST_TYPE,
          MEDIAN(LCL_LAB_TEST_VAL_TXT_FLOAT) as MEDIAN_TEST_RESULT,
          CASE WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT <15 THEN 5
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=15 AND MEDIAN_TEST_RESULT <30 THEN 4
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=30 AND MEDIAN_TEST_RESULT <60 THEN 3
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=60 AND MEDIAN_TEST_RESULT <90 THEN 2
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=90 AND MEDIAN_TEST_RESULT <=150 THEN 1
          WHEN TEST_TYPE = 'UACR' THEN NULL
          ELSE 0
          END AS GFR_STAGE,
          CASE WHEN TEST_TYPE = 'UACR' AND MEDIAN_TEST_RESULT <30 THEN 1
          WHEN TEST_TYPE = 'UACR' AND MEDIAN_TEST_RESULT >=30 AND MEDIAN_TEST_RESULT <300 THEN 2
          WHEN TEST_TYPE = 'UACR'AND MEDIAN_TEST_RESULT >=300 THEN 3
          WHEN MEDIAN_TEST_RESULT IS NULL THEN NULL
          WHEN TEST_TYPE = 'GFR' THEN NULL
          ELSE 0
          END AS UACR_STAGE
          FROM (
              SELECT DISTINCT CAST(CAST(PATIENT_ID AS INT) AS VARCHAR(64)) AS PATIENT_ID, NPI_ID , SVC_DT_STD, TEST_TYPE,
              TRY_TO_DOUBLE(TRIM(LCL_LAB_TEST_VAL_TXT)) AS LCL_LAB_TEST_VAL_TXT_FLOAT  FROM
              (
                SELECT *,
                CASE
                  WHEN UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%GFR%' OR UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%GLOMERULAR%' THEN 'GFR'
                  WHEN UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%ALB%' OR UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%RATIO%' THEN 'UACR'
                  ELSE 'OTHER'
                END AS TEST_TYPE,
                CASE
                  WHEN SVC_DT LIKE '%/%/%' THEN TO_DATE(SVC_DT,'MM/DD/YYYY')
                  ELSE TO_DATE(LEFT(SVC_DT,9),'DDMONYYYY')
                END AS SVC_DT_STD
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST
                WHERE REGEXP_LIKE(PATIENT_ID, '^[0-9]+$')
                AND LCL_LAB_TEST_VAL_TXT IS NOT NULL
                AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%-%'
                AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%>%'
                AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%<%'
                AND REGEXP_LIKE(TRIM(LCL_LAB_TEST_VAL_TXT), '^[0-9]+(\\.[0-9]+)?$')
              ) lab1
              LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR dct
              ON lab1.NPI_ID = dct.NPI_NUMBER
              WHERE UPPER(REPLACE(LCL_LAB_TEST_RSLT_UOM_CD_CLEAN, ' ','')) IN
              ('ML/MIN/1.73M2', 'MCG/MGCREAT','MG/GCREAT','MG/GCR','MG/G{CREAT}')
              AND PRIMARY_SPECIALTY_CODE IN ('FM', 'FPG', 'GP', 'GPM', 'IFP', 'IM', 'IMG', 'IPM', 'NRP', 'PHA')
              AND TEST_TYPE <> 'OTHER'
              AND SVC_DT_STD >= '2021-11-30'
              AND NPI_ID IS NOT NULL
          ) LAB
      GROUP BY 1, 2, 3, 4
"""

df_lab = Get_Data_Snowflakes(lab_query)
df_lab = df_lab.filter(col("DATE") <= date_end)

# HCP universe (same as 3.0 00_Write_Input_Files)
_hcp_universe = (spark.read.format("delta").load(hcp_universe_path).select("PROVIDER_ID").distinct()
                 .join(spark.read.format("delta").load(provider_npi_bhid_mapping_path).select("PROVIDER_ID", "NPI_NUMBER"),
                       on="PROVIDER_ID", how="left"))
df_lab = df_lab.join(_hcp_universe, how="inner", on="NPI_NUMBER")

# --- original 3.1 logic from here ---
df_lab = df_lab.withColumnRenamed("DATE", "SVC_DT")

# Filter to data >= DATA_START_DATE (from config)
df_lab = df_lab.filter(F.col("SVC_DT") >= DATA_START_DATE)

# Validation filter: 10 test HCPs (from config)
df_lab, test_patients = apply_hcp_filter(df_lab)

df_lab = df_lab.select("PATIENT_ID", "SVC_DT", "GFR_STAGE", "UACR_STAGE")

df_lab = df_lab.withColumn("PATIENT_ID", col("PATIENT_ID").cast('string'))

df_lab.display()

## 2. Data cleanup and processing

In [0]:
# Cohort-aware month-grain cleanup with forward-fill

import sys
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col

# Find the end date limit
end_date_limit = str(df_lab.agg({"SVC_DT": "max"}).collect()[0]['max(SVC_DT)'])

# Define forward fill windows (cohort-month-based)
window_month = Window.partitionBy('PATIENT_ID').orderBy('COHORT_MONTH').rowsBetween(-sys.maxsize, 0)

# Get LAST_TEST_DATE per patient
df_lab_with_last_date = df_lab.withColumn(
    "LAST_TEST_DATE", 
    F.lead("SVC_DT", 1).over(Window.partitionBy("PATIENT_ID").orderBy("SVC_DT"))
).withColumn("LAST_TEST_DATE", F.coalesce(col("LAST_TEST_DATE"), F.to_date(F.lit(end_date_limit))))

# Generate cohort-month sequence from SVC_DT to LAST_TEST_DATE
# Uses assign_cohort_month() from config to shift by offset
df_lab_with_month_seq = df_lab_with_last_date.withColumn(
    "MONTH_SEQ",
    F.expr(f"""
        sequence(
            {_cohort_month_sql('SVC_DT')},
            {_cohort_month_sql('LAST_TEST_DATE')},
            interval 1 month
        )
    """)
)

# Explode the month sequence to get each cohort-month in the range
df_lab_expanded = df_lab_with_month_seq.withColumn(
    "COHORT_MONTH",
    F.explode("MONTH_SEQ")
).drop("MONTH_SEQ")

# Forward-fill test stages across cohort-months
filled_gfr = F.last(col("GFR_STAGE"), ignorenulls=True).over(window_month)
filled_uacr = F.last(col("UACR_STAGE"), ignorenulls=True).over(window_month)

df_lab_filled = df_lab_expanded.withColumn("GFR_STAGE", filled_gfr).withColumn("UACR_STAGE", filled_uacr)

# Aggregate to remove duplicates per patient-cohort-month and take the max stages
df_lab_clean = df_lab_filled.groupBy("PATIENT_ID", "COHORT_MONTH").agg(
    F.max("GFR_STAGE").alias("GFR_STAGE"),
    F.max("UACR_STAGE").alias("UACR_STAGE")
).orderBy("PATIENT_ID", "COHORT_MONTH")

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
df_lab_clean = df_lab_clean.join(
    df_cohort_ref.select("cohort_ref_month", "cohort_label"),
    df_lab_clean.COHORT_MONTH == df_cohort_ref.cohort_ref_month,
    "left"
).drop("cohort_ref_month")

df_lab_clean.display()

In [0]:
df_lab_clean = df_lab_clean.withColumnRenamed("COHORT_MONTH", "DATE")

In [0]:
df_lab_clean.count(), len(df_lab_clean.columns)

In [0]:
# ============================================================
# COMPARISON: offset=0 validation vs production (same patients)
# ============================================================
base = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output"
import os as _os
_prod_path = f"{base}/kerendia_nba_3_0_lab_journey_output"

def _compare_with_prod():
  prod_lab = spark.read.format("delta").load(_prod_path)
  val_patients = [r.PATIENT_ID for r in df_lab_clean.select("PATIENT_ID").distinct().collect()]
  prod_lab_f = prod_lab.filter(col("PATIENT_ID").isin(val_patients))
  _run_comparison(prod_lab_f)

def _run_comparison(prod_lab_f):
  exec(_COMPARISON_BODY, globals(), {"prod_lab_f": prod_lab_f})

_COMPARISON_BODY = '''
val_count = df_lab_clean.count()
prod_count = prod_lab_f.count()
print(f"Validation (offset=0): {val_count:,}")
print(f"Production (same pts): {prod_count:,}")
print(f"Difference:            {val_count - prod_count:+,}")

# Key comparisons
val_keys = df_lab_clean.select("PATIENT_ID", "DATE")
prod_keys = prod_lab_f.select("PATIENT_ID", "DATE")

only_in_val = val_keys.subtract(prod_keys)
only_in_prod = prod_keys.subtract(val_keys)
print(f"\nRows only in validation: {only_in_val.count()}")
print(f"Rows only in production: {only_in_prod.count()}")

# Attribute: how many prod-only rows are before DATA_START_DATE?
prod_only_before = only_in_prod.filter(col("DATE") < DATA_START_DATE).count()
prod_only_after = only_in_prod.filter(col("DATE") >= DATA_START_DATE).count()
print(f"  Prod-only before {DATA_START_DATE}: {prod_only_before} (DATA_START_DATE filter)")
print(f"  Prod-only on/after {DATA_START_DATE}: {prod_only_after}")

# Stage differences on overlapping rows
common = df_lab_clean.alias("v").join(
    prod_lab_f.alias("p"),
    (col("v.PATIENT_ID") == col("p.PATIENT_ID")) & (col("v.DATE") == col("p.DATE")),
    "inner"
)
stage_diffs = common.filter(
    (col("v.GFR_STAGE") != col("p.GFR_STAGE")) |
    (col("v.UACR_STAGE") != col("p.UACR_STAGE")) |
    (col("v.GFR_STAGE").isNull() != col("p.GFR_STAGE").isNull()) |
    (col("v.UACR_STAGE").isNull() != col("p.UACR_STAGE").isNull())
)
print(f"\nOverlapping rows: {common.count():,}")
print(f"Stage value differences: {stage_diffs.count()}")
if stage_diffs.count() > 0:
    print("\nSample diffs:")
    stage_diffs.select(
        col("v.PATIENT_ID"), col("v.DATE"),
        col("v.GFR_STAGE").alias("val_GFR"), col("p.GFR_STAGE").alias("prod_GFR"),
        col("v.UACR_STAGE").alias("val_UACR"), col("p.UACR_STAGE").alias("prod_UACR")
    ).orderBy("PATIENT_ID", "DATE").show(20, truncate=False)
'''

# Optional check: the 3.0 production lab journey table may not exist after a data refresh
if _os.path.exists(_prod_path):
  _compare_with_prod()
else:
  print(f"SKIP comparison: production table not found at {_prod_path}")

In [0]:
df_lab.count(), len(df_lab.columns)

In [0]:


# VALIDATION: write to a separate path so we don't overwrite production data
df_lab_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_lab_journey_validation")